# trt_yolo_bench Colab runner

Run this notebook only in Google Colab with a GPU runtime. The first cell deliberately stops on a local CPU runtime so later cells do not produce misleading cascaded errors.

In [1]:
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA runtime detected. Open this notebook in Google Colab, "
        "then choose Runtime > Change runtime type > T4 GPU and reconnect."
    )
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)
print(sys.version)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [2]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/chao-dotcom/trt_yolo_bench.git"
REPO_DIR = Path("/content/trt_yolo_bench")
if not REPO_DIR.exists():
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if not REPO_DIR.exists():
        raise RuntimeError("Repository clone failed")
os.chdir(REPO_DIR)
print(Path.cwd())
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
required = ['--insecure-downloads', 'cuda.bindings']
source = Path('export/download_coco_slice.py').read_text() + Path('export/calibrator.py').read_text()
missing = [item for item in required if item not in source]
if missing:
    raise RuntimeError(f'GitHub checkout is stale; push the local fixes before running Colab. Missing: {missing}')

/content/trt_yolo_bench


## Install dependencies

The old notebook installed `pycuda` in the same pip command as TensorRT. On current Colab Python 3.13 images, `pycuda` has no compatible wheel and that makes pip report TensorRT as missing too. This project now supports `cuda-python`, so install the packages separately.

In [3]:
!python -m pip install -q --upgrade pip
!apt-get update -qq
!apt-get install -y -qq cmake libopencv-dev
!python -m pip install -q ultralytics onnx pycocotools requests opencv-python numpy
!python -m pip install -q tensorrt cuda-python
!python -m pip install --no-cache-dir -q --extra-index-url https://pypi.nvidia.com "nvidia-modelopt[all]"
!python -c "import tensorrt as trt; print('TensorRT', trt.__version__)"
!python -c "from cuda.bindings import driver; print('cuda-python import OK')"
!test -d /content/TensorRT || git clone --depth 1 -q https://github.com/NVIDIA/TensorRT.git /content/TensorRT

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../000-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../001-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../002-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../003-libxtst6_2%3a1.2.3-1.1ubuntu0.1_amd64.deb ...
Unpacking libxtst6:amd

In [4]:
from pathlib import Path
import os
import tensorrt_libs
headers = list(Path('/usr/local').glob('**/NvInfer.h')) + list(Path('/usr/include').glob('**/NvInfer.h'))
source_headers = Path('/content/TensorRT/include')
if not source_headers.joinpath('NvInfer.h').exists() and not headers:
    raise RuntimeError('TensorRT C++ headers are unavailable')
os.environ['TENSORRT_INCLUDE_DIR'] = str(source_headers if source_headers.joinpath('NvInfer.h').exists() else headers[0].parent)
os.environ['TENSORRT_LIB_DIR'] = str(Path(tensorrt_libs.__file__).parent)
print('TensorRT include:', os.environ['TENSORRT_INCLUDE_DIR'])
print('TensorRT libraries:', os.environ['TENSORRT_LIB_DIR'])

TensorRT include: /content/TensorRT/include
TensorRT libraries: /usr/local/lib/python3.13/dist-packages/tensorrt_libs


In [5]:
!python export/download_coco_slice.py --val-count 1000 --calib-count 500 --insecure-downloads

usage: download_coco_slice.py [-h] [--data-dir DATA_DIR]
                              [--val-count VAL_COUNT]
                              [--calib-count CALIB_COUNT] [--timeout TIMEOUT]
download_coco_slice.py: error: unrecognized arguments: --insecure-downloads


In [6]:
!python export/export_onnx.py --weights yolo11n.pt --out-dir models

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from 'yolo11n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.4 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 394ms
Prepared 3 packages in 825ms
Installed 3 packages in 20ms
 + colorama==0.4.6
 + onnxruntime==1.

In [7]:
!python export/convert_fp16.py --input models/yolo11n.onnx --output models/yolo11n_fp16.onnx
!python export/build_engines.py --onnx models/yolo11n.onnx --fp16-onnx models/yolo11n_fp16.onnx --engines-dir engines 2>&1 | tee build.log
from pathlib import Path
for name in ('yolo11n_fp32.engine', 'yolo11n_fp16.engine'):
    if not (Path('engines') / name).exists():
        raise RuntimeError(f'Missing engine after build: {name}')
if not Path('engines/yolo11n_int8.engine').exists():
    print('INT8 engine not produced: TensorRT 11 requires explicit Q/DQ quantization.')

Traceback (most recent call last):
  File "/content/trt_yolo_bench/export/build_engines.py", line 13, in <module>
    import pycuda.autoinit  # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'pycuda'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/trt_yolo_bench/export/build_engines.py", line 15, in <module>
    raise RuntimeError(
        "pycuda.autoinit import failed. Install pycuda in the CUDA environment before building engines."
    ) from exc
RuntimeError: pycuda.autoinit import failed. Install pycuda in the CUDA environment before building engines.


RuntimeError: Missing engine after build: yolo11n_fp32.engine

In [ ]:
!cmake -S harness -B harness/build
!cmake --build harness/build --parallel
!test -x harness/build/trt_yolo_bench

In [ ]:
!./harness/build/trt_yolo_bench engines/yolo11n_fp32.engine data/val_slice.json --out results_fp32.json --latency-out latency_fp32.csv 2>&1 | tee bench_fp32.log
!./harness/build/trt_yolo_bench engines/yolo11n_fp16.engine data/val_slice.json --out results_fp16.json --latency-out latency_fp16.csv 2>&1 | tee bench_fp16.log
!./harness/build/trt_yolo_bench engines/yolo11n_int8.engine data/val_slice.json --out results_int8.json --latency-out latency_int8.csv 2>&1 | tee bench_int8.log

In [ ]:
!python eval/run_ultralytics_reference.py --weights yolo11n.pt --out results_pytorch.json
!python eval/score_coco.py --dets results_pytorch.json | tee score_pytorch.log
!python eval/score_coco.py --dets results_fp32.json | tee score_fp32.log
!python eval/score_coco.py --dets results_fp16.json | tee score_fp16.log
!python eval/score_coco.py --dets results_int8.json | tee score_int8.log

In [ ]:
!mkdir -p results
!bash -lc 'nvidia-smi > results/env.txt; nvcc --version >> results/env.txt; python -c "import tensorrt as trt; print(trt.__version__)" >> results/env.txt; pip freeze >> results/env.txt'
from google.colab import files
import shutil
shutil.make_archive('/content/trt_yolo_bench-results', 'zip', '/content/trt_yolo_bench')
files.download('/content/trt_yolo_bench-results.zip')